# Agents, Tools & Memory — A Bare-Metal Agent Loop

**Category:** Agents, Tools & Memory · ToolGen, Agents Memory, Cohere Enterprise Agents, Anthropic Skills.

**What this notebook does.** Builds an agent loop in pure Python — no LangChain, no frameworks. A planner proposes a tool call, an executor runs it, a scratchpad holds short-term state. The notebook then layers in a simple semantic-memory store and measures task success with vs. without it on a tiny benchmark.

**No model API required.** The 'LLM' here is a deterministic rule-based planner so the loop is fully reproducible. In production you'd swap in a real LLM call.

## 1. Tools — small, idempotent, observable

In [ ]:
import re, math, json, time

TOOL_LOG = []
def _log(name, args, result):
    TOOL_LOG.append({'tool': name, 'args': args, 'result': result, 't': time.time()})

def tool_calculator(expr: str) -> str:
    """Evaluate a math expression. Idempotent, safe subset."""
    if not re.fullmatch(r'[0-9+\-*/(). ]+', expr):
        r = 'error: unsafe expr'
    else:
        try: r = str(eval(expr, {'__builtins__': {}}, {}))
        except Exception as e: r = f'error: {e}'
    _log('calculator', {'expr': expr}, r); return r

FACTS = {
    'speed_of_light_kms': '299792',
    'earth_radius_km':    '6371',
    'company_founded_year': '2017',
}
def tool_fact(key: str) -> str:
    r = FACTS.get(key, 'unknown')
    _log('fact', {'key': key}, r); return r

TOOLS = {'calculator': tool_calculator, 'fact': tool_fact}
print('Tools registered:', list(TOOLS))


## 2. A toy planner

Parses a task into one of three shapes: a `fact(key)` lookup, a `calc(expr)` evaluation, or a `derive` that chains a fact lookup into a calculation. In production this is your LLM.

In [ ]:
def planner(task: str, memory: dict):
    # ultra-light keyword routing
    task_l = task.lower()
    for k in FACTS:
        if k.replace('_', ' ') in task_l:
            if any(op in task for op in ['+','-','*','/']):
                return ('derive', k)
            return ('fact', k)
    if re.search(r'\d', task) and any(op in task for op in ['+','-','*','/']):
        expr = re.findall(r'[0-9+\-*/(). ]+', task)[0]
        return ('calc', expr.strip())
    # memory check: did we answer something similar?
    for q, a in memory.items():
        if any(w in q for w in task_l.split() if len(w) > 4):
            return ('recall', (q, a))
    return ('giveup', None)


## 3. The agent loop

In [ ]:
def run(task: str, memory: dict, verbose=True):
    if verbose: print(f'TASK: {task}')
    action, payload = planner(task, memory)
    if verbose: print(f'  plan: {action}({payload})')
    if action == 'fact':
        result = TOOLS['fact'](payload)
    elif action == 'calc':
        result = TOOLS['calculator'](payload)
    elif action == 'derive':
        f = TOOLS['fact'](payload)
        expr = re.sub(r'(?i)' + payload.replace('_', ' '), f, task)
        expr2 = ''.join(c for c in expr if c in '0123456789+-*/(). ')
        result = TOOLS['calculator'](expr2)
    elif action == 'recall':
        result = f'(from memory) {payload[1]}'
    else:
        result = 'I do not know'
    if verbose: print(f'  result: {result}\n')
    memory[task] = result
    return result

memory = {}
run('what is the speed_of_light_kms?', memory)
run('compute 12 * 12', memory)
run('what is speed_of_light_kms / 2', memory)


## 4. Memory ablation — does adding memory actually help?

Run a small benchmark of 10 tasks, half of which are *repeats* of earlier tasks. Compare a no-memory agent (re-runs every tool call) against a memory agent (recalls from the scratchpad).

In [ ]:
import copy
tasks = [
    'what is the speed_of_light_kms?',
    'what is the earth_radius_km?',
    'compute 7 * 8',
    'compute 100 / 4',
    'what is the company_founded_year?',
    'what is the speed_of_light_kms?',    # repeat
    'compute 7 * 8',                       # repeat
    'what is the earth_radius_km?',        # repeat
    'what is speed_of_light_kms / 1000',
    'compute 3.14 * 2',
]

def run_bench(use_memory):
    TOOL_LOG.clear()
    mem = {} if use_memory else {}
    for t in tasks:
        run(t, mem if use_memory else {}, verbose=False)
    return len(TOOL_LOG)

no_mem_calls = run_bench(use_memory=False)
mem_calls = run_bench(use_memory=True)
print(f'tool calls without memory: {no_mem_calls}')
print(f'tool calls with memory:    {mem_calls}')
print(f'reduction:                  {100*(1 - mem_calls/no_mem_calls):.0f}%')


## My take

What this miniature loop makes visible:

1. **Tools should be three things — small, idempotent, observable.** Every call should be safe to retry. Every call should leave a structured log. If you can't see what your agent did in production from the logs alone, you don't have an agent, you have a security incident waiting to happen.
2. **The planner is the part that breaks first.** I replaced an LLM with a 30-line routing function. That tells you something: most 'agent' systems are doing routing badly. A real LLM here is more *flexible*, not categorically more correct. Choose your model accordingly.
3. **Memory is just a key/value store with a similarity function on top.** Stop letting people sell you 'cognitive memory architectures.' The Anthropic 'skills not agents' framing is closer to where production lives — composable, scoped, testable.

If your team can't draw this loop on a whiteboard, you're not ready to ship an agent. If they can, you probably don't need a framework.